# 02 — Bayesian Logistic Regression (GPU/CUDA)

Specify priors based on domain knowledge, build the generative model in PyMC, and run MCMC sampling using **NumpyPro + JAX on your NVIDIA GPU**.

## 0. GPU setup check

Run this first to confirm JAX can see your GPU before sampling.

In [9]:
import jax, jaxlib

print(f"JAX version    : {jax.__version__}")
print(f"JAXlib version : {jaxlib.__version__}")
print(f"Devices        : {jax.devices()}")

# Should print something like:
# Devices: [CudaDevice(id=0)]
# If it shows CpuDevice, JAX is not using your GPU — check your jax[cuda12] install.

JAX version    : 0.4.38
JAXlib version : 0.4.38
Devices        : [CpuDevice(id=0)]


## 1. Prior elicitation

Before touching PyMC, we decide what we *believe* about each feature.

| Feature | Prior belief | Expected sign |
|---|---|---|
| tenure | Longer tenure → loyal customer | Negative |
| Contract: month-to-month | No lock-in → easy to leave | Positive |
| MonthlyCharges | Higher bill → more incentive to shop around | Positive |
| Contract: two-year | Locked in → unlikely to churn | Negative |
| Fiber optic | Competitive market, higher bills | Positive |
| SeniorCitizen | Slightly more vulnerable to churn | Positive |

All betas: `Normal(0, 1)` — weakly informative on a standardised scale.

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df = pd.get_dummies(df, columns=["Contract", "InternetService"], drop_first=False)
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract_Month_to_month",
    "Contract_Two_year",
    "InternetService_Fiber_optic",
    "SeniorCitizen",
]

X = StandardScaler().fit_transform(df[features])
y = df["Churn"].values
print(f"X: {X.shape}, y: {y.shape}, churn rate: {y.mean():.3f}")

X: (7032, 7), y: (7032,), churn rate: 0.266


## 2. Build the model

In [3]:
with pm.Model() as churn_model:
    # Priors — weakly informative
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=X.shape[1])

    # Linear combination → probability via sigmoid
    logit_p = alpha + pt.dot(X, betas)
    p = pm.Deterministic("p", pm.math.sigmoid(logit_p))

    # Likelihood
    observed = pm.Bernoulli("churn", p=p, observed=y)

try:
    pm.model_to_graphviz(churn_model)
except Exception:
    print("graphviz not installed — skipping model graph.")
    print("Model: alpha (intercept) + betas (7 features) -> sigmoid -> Bernoulli")

## 3. Sample the posterior on GPU


In [6]:
import os
os.makedirs("../outputs", exist_ok=True)

with pm.Model() as churn_model_gpu:
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=X.shape[1])

    logit_p = alpha + pt.dot(X, betas)
    # ← No pm.Deterministic("p", ...) here — that's what caused OOM
    observed = pm.Bernoulli("churn", p=pm.math.sigmoid(logit_p), observed=y)

    trace = pm.sample(
        2000, tune=1000, chains=4,
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=True,
        nuts_sampler="numpyro",
    )

az.to_netcdf(trace, "../outputs/trace.nc")
print("Trace saved.")

sample: 100%|██████████| 3000/3000 [00:10<00:00, 277.93it/s, 15 steps of size 1.69e-01. acc. prob=0.93]


Trace saved.


In [7]:
# Compute p on CPU from saved posterior samples — no GPU memory needed
betas_flat = trace.posterior["betas"].values.reshape(-1, 7)  # (4000, 7)
alpha_flat  = trace.posterior["alpha"].values.reshape(-1)    # (4000,)

logit_samples = alpha_flat[:, None] + betas_flat @ X.T       # (4000, N)
p_samples     = 1 / (1 + np.exp(-logit_samples))             # (4000, N)
mean_p        = p_samples.mean(axis=0)                        # (N,)

print(f"Mean churn probability: {mean_p.mean():.3f}")
print(f"High risk customers (>0.5): {(mean_p > 0.5).sum()}")

Mean churn probability: 0.266
High risk customers (>0.5): 1444


## 4. Posterior summary

In [8]:
summary = az.summary(trace, var_names=["alpha", "betas"])
summary.index = ["alpha"] + features
summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]]

,mean,sd,hdi_3%,hdi_97%,r_hat
alpha,-1.659,0.052,-1.760,-1.566,1.0
tenure,-1.245,0.141,-1.521,-0.990,1.0
MonthlyCharges,0.384,0.076,0.246,0.529,1.0
TotalCharges,0.428,0.146,0.147,0.699,1.0
Contract_Month_to_month,0.458,0.052,0.361,0.555,1.0
Contract_Two_year,-0.383,0.073,-0.523,-0.244,1.0
InternetService_Fiber_optic,0.354,0.058,0.241,0.461,1.0
SeniorCitizen,0.153,0.029,0.096,0.206,1.0
